#### Introducción:
Durante el curso de la historia, los seres vivos hemos tenido que enfrentar todo tipo de adversidades y adaptarnos a un ambiente que con frecuencia ha ido cambiando. Esto ha sido posible gracias a la selección natural, que ha mantenido viva la llama de la vida durante millones de años. Una de las adaptaciones más valiosas que adquirimos los animales fue el sistema nervioso: un conjunto de esatructuras y circuitos neuronales interconectados capaces de capturar lo que sucede a nuestro alrededor, integrar esas señales del exterior y procesarlas para generar una respuesta.

Podría decirse que la función del sistema nervioso es interpretar una serie de estímulos (inputs) que le llegan del exterior para generar una respuesta (output) que se ejecuta en el organismo. Es por ello que las diferentes funciones y estructuras que componen el sistema nervioso pueden simularse en un ordenador. De esto se encarga la neurociencia computacional.

Este programa pretende simular el tacto (mecanorrecepción). Para ello, se emulará una porción de piel de 10cm con 100 mecanorreceptores. Estos mecanorreceptores van a recoger los diferentes estímulos (pinchazos) que ejecutemos sobre la piel, recogiendo información sobre la presión aplicada, la posición del pinchazo y la anchura del mismo. Estos datos serán transmitidos a una red de neuronas que afinarán la señal (inhibición lateral) y se desensibilizarán conforme vayamos generando estímulos en la piel (habituación). Por otra parte, los estímulos con una presión alta serán procesados como dañinos en un proceso conocido como nocicepción (es decir, la percepción del dolor).

#### Librerías:
En este trabajo se hará uso de tres librerías: **numpy** y **scipy** para realizar operaciones y funciones matemáticas, y **matplot** para la representación gráfica de los distintos procesos y señales generadas. Para el correcto funcionamiento del programa, el usuario deberá tener estas librerías previamente instaladas.
Para el procesamiento de los estímulos (inhibición lateral y habituación) se usará la función **convolve** (disponible en scipy).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp  
from scipy.signal import convolve
print(f"NumPy versión {np.__version__} cargado correctamente.")
print(f"MatPlot versión {plt.matplotlib.__version__} cargado correctamente.")
print(f"SciPy versión {sp.__version__} cargado correctamente.")
print( "Función convolve lista para usar.")

NumPy versión 2.4.4 cargado correctamente.
MatPlot versión 3.10.8 cargado correctamente.
SciPy versión 1.17.1 cargado correctamente.
Función convolve lista para usar.


#### Clases:
Para llevar estos procesos a un programa de Python, se han creado tres clases distintas, que corresponderían a los tres acontecimientos que ocurren durante la codificación del tacto en el sistema nervioso:
    1. La clase **estimulo** sería la encargada de representar la piel y los distintos estímulos que ejecutemos sobre ella.
    2. La clase **red** correspondería a la red neuronal responsable de codificar el tacto, ejerciendo sobre la señal recibida los mecanismos de inhibición lateral y habituación.
    3. La clase **nocicepción** actuará cuando un estímulo supere el umbral de dolor que hayamos asignado a la piel, convirtiéndose en un estímulo dañino. Esta clase está separada de la anterior (red) porque en el sistema nervioso las neuronas encargadas de la nocicepción (fibras Aδ y fibras C) forman una vía distinta a las neuronas encargadas de la mecanorrecepción "normal" (fibras A y B). Además, este proceso no está sometido a habituación ya que su función es informar sobre aquellos pinchazos que pueden ser dañinos. 

#### Estímulo:
Esta clase codificará la piel sobre la que el programa va a trabajar (self.piel), con una longitud de 100mm y 100 receptores distribuidos por igual en toda su longitud; así como los estímulos que se ejecuten sobre ella (que quedarán definidos en la función **pinchazo**).
El pinchazo dependerá de tres variables cuyo valor será el que el usuario le asigne:
    La **presión** será la fuerza con la que se ejecute el pinchazo, deberá ser entre 0 y 100. Si el valor introducido es menor que 5, no se disparará ninguna señal (esto sería el umbral de disparo de la señal, que en el sistema nervioso está para eliminar señales innecesarias o que considera poco importantes). Esta variable será importante tanto para codificar la nocicepción como para representaciones gráficas posteriores, por lo que se guardará en otra variable adicional (self.evaluar_presion).
    La **posición** será dónde se va a aplicar el pinchazo, esta piel está representada en una dimensión, por lo que también deberá ser entre 0 (a la izquierda) y 100 (a la derecha). Esta variable será importante para generar gráficas adicionales, por lo que se guardará en una variable adicional (self.evaluar_posicion).
    El **ancho** es la porción de piel que se va a estimular, pudiendo ser entre 0 y 100. Al igual que con la presión, la piel tiene un umbral de discriminación táctil a partir del cual no se distingue la distancia entre dos estímulos (o, en este caso, la anchura del pinchazo). Por ello, todos los estímulos con una anchura menor que 5mm automáticamente se codificarán como estímulos de 5mm.
El pinchazo deformará la piel curvándola. Dicha curvatura se representará mediante una función gaussiana que dependerá de los parámetros introducidos. Esto se guardará en la variable self.touch.
Por último, la función **graficaest** codificará el estímulo (la curvatura generada en la piel, representada en azul).

In [2]:
class estimulo:
    def __init__(self): 
        self.piel= np.linspace(0, 100, 100) # Este modelo de piel tiene una longitud de 100 mm y 100 receptores.
        self.touch= None
        self.evaluar_presion= None
        self.evaluar_posicion= None          
    def pinchazo(self):
        while True:
            try:
                presion= float(input("Presión del estímulo (de 0 a 100): "))
                self.evaluar_presion= presion
                if 0 <= presion <= 100:
                    break 
                else:
                    print("Por favor, el número debe estar entre 0 y 100.")
            except ValueError:
                print("Entrada no válida. Por favor, introduce un número (usa punto para decimales).")
        while True:
            try:
                posicion= float(input("Posición del estímulo (de 0 a 100): "))
                self.evaluar_posicion= posicion
                if 0 <= posicion <= 100:
                    break 
                else:
                    print("Por favor, el número debe estar entre 0 y 100: ")
            except ValueError:
                print("Entrada no válida. Por favor, introduce un número (usa punto para decimales).")
        while True:
            try:
                ancho= float(input("Ancho del estímulo (de 0 a 100): "))
                if 0 <= ancho <= 100:
                    break
                else:
                    print("Por favor, el número debe estar entre 0 y 100: ")
            except ValueError:
                print("Entrada no válida. Por favor, introduce un número (usa punto para decimales).")
        if presion < 5:
            presion= 0 # Para eliminar señales innecesarias, como estímulos con una presión insignificante o "ruido", la piel tiene un umbral de disparo de señal.
        if ancho < 5:
            ancho= 5 # La piel tiene un umbral de discriminación táctil de 5mm.
        self.touch= presion*np.exp(-0.5*((self.piel-posicion)/ancho)**2) # El estímulo táctil se representa mediante una función gaussiana.
        print("¡Estímulo generado!") 
        return self.touch
    def graficaest(self):
        x=self.piel
        plt.plot(x, -1*self.touch, label="Estímulo Táctil", color="blue", lw=2)

#### Red:
Esta clase representará a la red de neuronas implicadas en transformar el estímulo que la piel ha recibido en señales eléctricas que el sistema nervioso central pueda interpretar. En biología, a este proceso se le conoce como mecanotransducción.
El estímulo captado por los receptores de la piel será sometido a inhibición lateral y habituación mediante la función **procesar_señal**. 
La **inhibición lateral** es un proceso que ocurre durante la mecanotransducción mediante el cual las neuronas del "centro" (más próximas al estímulo) lanzan señales inhibitorias a las neuronas circundantes, reduciendo la señal de estas últimas. Como consecuencia de este proceso, la curva generada en la piel inicialmente se afilará en la red, siendo más estrecha y localizando mejor el centro de la estimulación. El programa reproduce este fenómeno con la variable **self.inhibicion_lat**, que a través de la convolución aplicará la inhibición lateral sobre la deformación de la piel provocada por el pinchazo.
Por otra parte, la **habituación** es un fenómeno que ocurre cuando la exposición a un estímulo es repetida. Conforme la piel se ve sometida al mismo estímulo, la respuesta a este baja. Esto ocurre por el mismo motivo que los umbrales de disparo y discriminación táctil: no interesa procesar los estímulos que no son importantes (el cerebro busa optimizar el procesamiento de señales). La habituación es la responsable de que no notemos constantemente el contacto con la ropa que llevamos, nuestro pelo o pendientes. La habituación en este modelo está codificada en la variable **self.factor_habituación**, que empieza siendo 1 pero va disminuyendo un 5% con cada repetición hasta llegar a un tope (0.1) a partir del cual deja de disminuir. El factor de habituación va cambiando con cada repetición al ser dependiente de la variable **self.contadorest**, que guarda el número de pinchazos que el usuario ha hecho.
Una vez ejecutada la función procesar_señal, el resultado (**self.resultado_final**) quedará guardado en la lista **self.historial_pinchazos**, que guardará mediante una serie de diccionarios las distintas características de cada estímulo (se guardará la presión recibida, la posición del pinchazo, la señal generada en la red y el valor del factor de habituación para cada pinchazo). Esto funcionaría como una versión simplificada de la propia memoria de los seres vivos.
El programa generará una gráfica de la señal generada con la función **graficared**, que aparecerá de color verde. Es importante tener clara la diferencia entre el pinchazo generado en la piel (el estímulo que recibe) y la señal generada por la red (lo que el sistema nervioso interpreta).
La función **resetear_sistema** devuelve el factor de habituación a su valor original (1) en caso de que el usuario lo necesite (no responde necesariamente a ningún fenómeno biológico real).
Finalmente, en caso de que el usuario desee ver un resumen (es decir, "recordar" las señales generadas), podrá ejecutar la función **mostrar_resumen**, donde además de las señales generadas podrá ver dónde se ha aplicado el estímulo (para, como se explicará a continuación, saber si los estímulos fueron dolorosos o no).

In [ ]:
class red:
    def __init__(self):
        self.inhibicion_lat= np.array([0.9, 1.5, -6, 1.5, 0.9])
        self.resultado_final= None 
        self.historial_pinchazos= [] # Historial con los pinchazos recibidos.
        self.contadorest= 0
    def procesar_señal(self, señal_input):
        self.contadorest= self.contadorest + 1
        if self.contadorest > 1:
            # Por cada repetición, la respuesta baja (habituación). Esto ocurre en la mecanorrecepción, no en la nocicepción (que en el sistema nervioso se procesa en una vía diferente).
            self.factor_habituacion= max(0.1, 1.0 - (self.contadorest - 1) * 0.05) # La sensibilidad disminuye un 5% con cada repetición, sin llegar ser más pequeña que un 10%.
            print("Habituación activada: Respuesta reducida al", self.factor_habituacion*100, "%")
        else:
            self.factor_habituacion= 1.0 # Empieza al 100% de sensibilidad.
        # Procesamiento con el factor de habituación:
        self.resultado_final= convolve(señal_input, self.inhibicion_lat, mode="same") * self.factor_habituacion # En esta línea del código se aplica tanto la inhibición lateral como la haituación.
        # Diccionarios para guardar el historial de señales generadas:
        self.historial_pinchazos.append({
            "contador": self.contadorest,
            "presion":piel.evaluar_presion,
            "posicion":piel.evaluar_posicion,
            "señal": self.resultado_final.copy(),
            "habituacion": self.factor_habituacion,
            })
        return self.factor_habituacion
    def graficared(self, x):
        plt.plot(x, self.resultado_final, label="Señal Procesada (Red)", color="green", lw=2)
    def resetear_sistema(self):
        self.factor_habituacion= 1
        self.historial_pinchazos= []
        self.contadorest= 0
    def mostrar_resumen(self):
        print("RESUMEN DE ACTIVIDAD:")                  
        for d in self.historial_pinchazos:
            if d["presion"]>66:
                tipo_señal= "Sí"
            else:
                tipo_señal= "No"
            print("Pinchazo:", d["contador"], "  Presión:",d["presion"], "  Sensibilidad:", d["habituacion"], "  Nocicepción:", tipo_señal)

#### Nocicepción:
La nocicepción en sí misma podría considerarse una función separada de la mecanorrecepción, aunque están estrechamente ligadas. Es por ello que en este programa se codifican en clases separadas. En este programa se ha simplificado para hacerla una señal binaria (o hay dolor o no lo hay) que se dispara si la presión del estímulo supera al umbral de dolor codificado en la variable **self.umbral_dolor**.
Se comparará la presión del estímulo con el umbral de dolor en la función **evaluar_daño**, si la presión es mayor que el umbral (66.0) se disparará la nocicepción (el estímulo le dolerá a la piel) y se enviará un mensaje de alerta (tal y como hace el sistema nervioso cuando nos hacemos daño). Además, las señales nociceptivas se guardarán en la lista **self.registro_alertas** y en la variable **self.contador_alertas**.
Por último, se graficarán aquellas señales nociceptivas con la función **graficar_alerta**, que en caso de haber nocicepción mostrrará el punto donde se ha aplicado el estímulo y un sombreado rojo por encima del umbral de dolor. El punto coincidirá con la deformación de la piel, no con la señal procesada por la red neuronal al ser dos procesos diferentes.

In [ ]:
class nocicepcion:
    def __init__(self):
        self.umbral_dolor= 66.0 # A partir de esta presión se generará una señal nociceptiva (es decir, que le provocará dolor a la piel).
        self.registro_alertas= [] # Historial de señales nociceptivas.
        self.estado_alerta= False
        self.contador_alertas= 0
    def evaluar_daño(self, presion_input):
        if presion_input > self.umbral_dolor:
            self.estado_alerta = True
            mensaje = f"¡ALERTA! Nociceptores activados. Presión de {presion_input} es peligrosa."
            print(mensaje)
            self.registro_alertas.append(mensaje) # Se añade al historial.
            self.contador_alertas= self.contador_alertas + 1
        else:
            self.estado_alerta= False
            print("Presión dentro del rango seguro (Mecanorrecepción pura).")
    def graficar_alerta(self): # Gráfica para la nocicepción:
        if self.estado_alerta:
            plt.axhspan(-1*self.umbral_dolor, -500, color="red", alpha=0.1, label="Zona de Dolor") # Sombreado de dolor.
            plt.scatter(piel.evaluar_posicion, -1*piel.evaluar_presion, color="red", s=30, zorder=5, label="Señal Nociceptiva") # Señal nociceptiva en el punto donde se ha aplicado el estímulo. 

#### Ejecución del programa:
Primero de todo, se hace un llamado a las clases a través de las instancias **piel** (estímulo), **red_neuronal** (red) y **sistema_dolor** (nocicepción).
A continuación, se ejecuta la función **pinchazo** para generar el estímulo, **procesar_señal** para aplicar la inhibición lateral y habituación, y **evaluar_daño** para disparar o no la señal nociceptiva.
Por último, se genera la gráfica a partir de las funciones para graficar cada una de las clases. Se añade a esta gráfica unificada una línea discontinua que marque el umbral nociceptivo y un título en donde se indica el número del estímulo y el valor del factor de habituación que éste tiene, así como una leyenda para interpretar las gráficas.

In [ ]:
# Instancias:
piel= estimulo()
red_neuronal= red()
sistema_dolor= nocicepcion()

# Generación del estímulo:
presion_usuario= piel.pinchazo()

# Procesamiento del estímulo por la red neuronal (inhibición lateral y habituación):
red_neuronal.procesar_señal(piel.touch)

# Procesamiento de dolor:
sistema_dolor.evaluar_daño(piel.evaluar_presion) 

# Gráfica:
plt.figure(figsize=(10, 6))
piel.graficaest() # Línea azul (estímulo).
red_neuronal.graficared(piel.piel) # Línea verde (señal generada por la red neuronal).
sistema_dolor.graficar_alerta() # Datos de la nocicepción (si aplica).
plt.axhline(-sistema_dolor.umbral_dolor, color="orange", linestyle="--", label="Umbral Nociceptivo") # Línea que marca el umbral de nocicepción.
plt.title(f"Estímulo nº {red_neuronal.contadorest}. Sensibilidad: {red_neuronal.factor_habituacion*100:.0f}%.")
plt.ylim(-225, 10)
plt.legend()
plt.show()

#### Ejecutar varios pinchazos, resetear la habituación, mostrar el resumen de los estímulos, o cerrar el programa:
Para poder observar fenómenos como la habituación es necesario que el programa funcione dentro de un bucle. Es por ello que cuando se termina de generar un pinchazo el usuario tiene la opción de generar un **nuevo pinchazo (opción 1)**. Si se elige esta opción, se volverá a ejecutar otra vez el proceso descrito anteriormente, con el valor que corresponda al factor de habituación para el nuevo pinchazo.,
Si se quiere resetear la habituación, el usuario podrá ejecutar la función **resetear_sistema** a través de la **opción 2**.
Para ver el resumen, el usuario tendrá a su disposición la **opción 3** para ejecutar la función **mostrar_resumen**. También se mostrará un recuento de todos los pinchazos dados y cuántos de estos fueron dolorosos.
Por último, cuando se quiera **cerrar el programa**, se podrá elegir la **opción 4**.

In [ ]:
while True:
    print("1. Generar nuevo estímulo (pinchazo).")
    print("2. Resetear habituación.")
    print("3. Ver historial de datos.")
    print("4. Salir.")
    
    opcion= input("Selecciona una opción: ")

    if opcion== "1":
        # Generación del estímulo:
        presion_leida = piel.pinchazo()
            
        # Procesamiento del estímulo por la red neuronal (inhibición lateral y habituación):
        h_actual = red_neuronal.procesar_señal(piel.touch)
            
        # Procesamiento del dolor:
        sistema_dolor.evaluar_daño(piel.evaluar_presion)
            
        # Gráfica:
        plt.figure(figsize=(10, 5))
        piel.graficaest() # Línea azul (estímulo).
        red_neuronal.graficared(piel.piel) # Línea verde (señal generada por la red neuronal).
        if piel.evaluar_presion> sistema_dolor.umbral_dolor: # Representación de la nocicepción (si hay):
            plt.axhspan(-1*sistema_dolor.umbral_dolor, -500, color="red", alpha=0.1, label="Zona Nociceptiva") # Sombreado de dolor.
            plt.scatter(piel.evaluar_posicion, -1*piel.evaluar_presion, color="red", s=30, zorder=5, label="Señal Nociceptiva") # Señal nociceptiva en el punto donde se ha aplicado el estímulo.
        plt.title(f"Estímulo nº {red_neuronal.contadorest}. Sensibilidad: {red_neuronal.factor_habituacion*100:.0f}%.")
        plt.axhline(-sistema_dolor.umbral_dolor, color="orange", linestyle="--", label="Umbral Nociceptivo")
        plt.ylim(-225, 10)
        plt.legend()
        plt.show()

    elif opcion== "2":  
        red_neuronal.resetear_sistema()

    elif opcion== "3":
        red_neuronal.mostrar_resumen()
        print("Total de señales generadas:",red_neuronal.contadorest, "  Total de señales nociceptivas:", sistema_dolor.contador_alertas)
        plt.figure(figsize=(10, 5))
        plt.axhline(-1*sistema_dolor.umbral_dolor, color="orange", linestyle="--", label="Umbral Nociceptivo")
        for dato in red_neuronal.historial_pinchazos:
            linea,=plt.plot(piel.piel, dato["señal"], label=f"Estímulo {dato["contador"]}") # Representación de las señales generadas por la red (no el estímulo).
            color_asignado = linea.get_color()
            plt.scatter(dato["posicion"], -1 * dato["presion"], color=color_asignado, s=30) # Representación del estímulo mediante un punto (así se puede evaluar si hubo nocicepción o no).   
        plt.title("Comparativa de todas las señales registradas.")
        plt.ylim(-225, 10)
        plt.legend()
        plt.show()

    elif opcion== "4":
        print("Cerrando el programa.")
        break

    else:
        print("Por favor, introduce una opción válida.")

#### Utilidad y posibles mejoras:
Este modelo de piel es muy simple todavía, por lo que no serviría como un modelo experimental complejo o totalmente fiable. Aun así, representa de una manera clara los principales procesos que ocurren durtante la codificación del tacto y el dolor, constituyendo una buena herramienta en ámbitos como el de la educación.
Cuando la estimulación se produce en los extremos de la piel, o cuando el pinchazo tiene una anchura notoria, la red neuronal puede tener errores en su respuesta. Si bien esto es algo a mejorar, podría tener su correlato biológico en las ilusiones sensoriales (e ilusiones ópticas en el caso de la visión), ya que estos fenómenos se dan cuando el sistema nervioso se "confunde" a la hora de interpretar un estímulo.
Otras posibles mejoras a tener en cuenta para el futuro pueden ser ampliar la piel a dos dimensiones para un mayor realismo o añadir complejidad a la nocicepción, ya sea separando los distintos tipos de dolor (como el agudo y el duradero, codificados en las fibras Aδ y fibras C respectivamente), o generar algún tipo de habituación al dolor modificando el umbral nociceptivo.